# YARG Resurrected: convert a batch of songs

Turns a folder of songs in your Google Drive into YARG song folders, on Colab's GPU:
guitar and bass charts, and a vocals part with lyrics timed word by word against the voice.

**Rap, R&B and other songs carried by the voice** (with MODE = split) are charted the way the
converter learned on nine real rap songs: the guitar follows the vocals as single notes, and the
instruments where they lead (a piano intro, a chime or keys loop, a break); the bass follows the
808, with each hit on a still-ringing 808 charted, double hits a 32nd note apart charted as two
notes, and its frets following the 808's actual note; the tempo is counted by the backbeat, so a
95 BPM song is not charted at 190. Every split song also gets **Deep Music Mode** parts: guitar
adds the vocals and the beat's melody all the way through, bass adds the drum hits. Turn on the
Deep Music Mode modifier in YARG to play them (no drum kit needed).

**Lyrics:** looked up online (LRCLIB, NetEase, Lyrics.ovh) and timed from the singer's voice;
for songs with no lyrics anywhere, written from the voice by speech-to-text (Whisper large-v3,
about 3 words in 10 wrong on rap, fewer on singing).

**Before you start:** Runtime > Change runtime type > **GPU** (T4 or better). Put your songs in
`My Drive/YARG/input` (any of mp3, wav, ogg, flac, m4a; loose, in subfolders, or zipped). From a phone:
select the songs in your Files app, Share > Drive, and pick that folder.

**Optional, per song:** a `songs.csv` in that folder with columns
`file,mode,notes,title,artist,album,year,genre,start,end` (mode: split, rnb or normal; notes: more,
standard or fewer; start and end in seconds, to cut a music video's intro or outro off). Leave
columns empty to have them filled in automatically: title, artist, album, year and genre come from
the file's tags, its name ("Artist - Title") and MusicBrainz. For example:

```
file,mode,notes,title,artist,album,year,genre,start,end
Girls_Generation______Gee_MV.mp3,split,more,Gee,Girls' Generation,,,,19.9,220.5
Seduction.mp3,rnb,,,Usher,,,,,
```

Then **Runtime > Run all**. Results go to `My Drive/YARG/output`.

**Optional, checking the converter:** put your copies of the test recordings in
`My Drive/YARG/test-songs` and run the last cell. Each is checked against what it is known to
produce (tempo, what the guitar follows, how much of the vocals, instruments, bass hits and drums
the charts cover, bass notes on time); see `converter/real-songs/README.md`. Run it after updating
the converter, to be sure nothing learned so far was lost.

In [ ]:
#@title Settings
SONGS = "/content/drive/MyDrive/YARG/input"   #@param {type:"string"}
OUTPUT = "/content/drive/MyDrive/YARG/output" #@param {type:"string"}
MODE = "split"   #@param ["split", "rnb", "normal"]
NOTES = "more"   #@param ["more", "standard", "fewer"]
JOBS = "auto"    #@param ["auto", "1", "2", "3", "4"]
#@markdown JOBS: songs converted at once. "auto" picks from the runtime's memory and processors (an A100 or L4 runtime: usually 3 to 4; a T4: 1).
BRANCH = "ccr-7c39c845-6cqp27" #@param {type:"string"}
#@markdown MODE: **split** separates the instruments (needed for the rap / vocal-led charts, the 808 bass and Deep Music Mode); rnb makes the guitar follow the lead singer's tune; normal charts the whole mix without separating (fast, no bass chart).
#@markdown BRANCH: the converter version. Each run of the setup cell updates to the newest commit on it.

In [ ]:
#@title 1. Connect Google Drive
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
os.makedirs(SONGS, exist_ok=True); os.makedirs(OUTPUT, exist_ok=True)
AUDIO = (".mp3", ".wav", ".ogg", ".opus", ".flac", ".m4a", ".aac", ".webm")
# The songs are gathered on the Colab disk: loose files, files in subfolders, and the insides of zips
os.makedirs("/content/songs", exist_ok=True)
for f in os.listdir("/content/songs"): os.remove(os.path.join("/content/songs", f))
for root, _, files in os.walk(SONGS):
    for f in files:
        path = os.path.join(root, f)
        if f.lower().endswith(AUDIO):
            shutil.copy(path, "/content/songs/" + f)
        elif f.lower().endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                for name in z.namelist():
                    base = os.path.basename(name)
                    if base.lower().endswith(AUDIO) and not base.startswith("."):
                        with z.open(name) as src, open("/content/songs/" + base, "wb") as dst:
                            shutil.copyfileobj(src, dst)
        elif f == "songs.csv":
            shutil.copy(path, "/content/songs/songs.csv")
songs = sorted(f for f in os.listdir("/content/songs") if f.lower().endswith(AUDIO))
print(len(songs), "songs found in", SONGS, "(loose, in subfolders or in zips)")
for s in songs[:40]: print("  ", s)


**2. Set up** (about 3 minutes, once per session)

In [ ]:
%%bash -s "$BRANCH"
set -e
cd /content
apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless ffmpeg > /dev/null
pip -q install faster-whisper num2words
if [ -d YARG_Resurrected/.git ]; then
  # Already here from an earlier run: brought up to the newest converter on the branch
  git -C YARG_Resurrected fetch -q --depth 1 origin "$1" && git -C YARG_Resurrected checkout -q -B "$1" FETCH_HEAD
else
  git clone -q --depth 1 -b "$1" https://github.com/ClockworkBlack/YARG_Resurrected.git
fi
echo "Converter version: $(git -C YARG_Resurrected log -1 --format='%h %cd %s' --date=short)"
mkdir -p models lib
cd models
[ -s htdemucs_6s.onnx ] || curl -sSL --fail -o htdemucs_6s.onnx "https://huggingface.co/StemSplitio/htdemucs-6s-onnx/resolve/49df9b6989cf2150840ea65b0bef77a2e471b678/htdemucs_6s_fp16weights.onnx"
[ -s wav2vec2.onnx ] || curl -sSL --fail -o wav2vec2.onnx "https://huggingface.co/Xenova/wav2vec2-base-960h/resolve/a19f851b3d42865797e410752b4c570c871e4825/onnx/model.onnx"
echo "7ce55792e2231c93fbf92de95f5fd5b3a5e6c89f7db690dfd693e8f1dce56869  htdemucs_6s.onnx" | sha256sum -c
echo "e46614273f03ff4b87923a965e417fa72004825522cb007c9c25633b8475490d  wav2vec2.onnx" | sha256sum -c
cd ../lib
[ -s onnxruntime_gpu.jar ] || curl -sSL --fail -o onnxruntime_gpu.jar "https://repo1.maven.org/maven2/com/microsoft/onnxruntime/onnxruntime_gpu/1.31.1/onnxruntime_gpu-1.31.1.jar"
[ -s json.jar ] || curl -sSL --fail -o json.jar "https://repo1.maven.org/maven2/org/json/json/20240303/json-20240303.jar"
cd /content/YARG_Resurrected/converter
S=app/src/main/java/com/clockworkblack/yargconverter
rm -rf /content/build && javac -encoding UTF-8 -nowarn -cp /content/lib/onnxruntime_gpu.jar:/content/lib/json.jar -d /content/build \
  $(find $S -path '*core*' -name '*.java') $S/stems/*.java $S/LrcLib.java $S/NetEase.java $S/MusicBrainz.java \
  desktop/com/clockworkblack/yargconverter/Lookup.java core-test/MidiCheck.java desktop/ConvertSong.java 2>&1 | grep -v "^Note" || true
echo "Ready."
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime > Change runtime type > GPU"

In [ ]:
#@title 3. Convert
import os, subprocess
env = dict(os.environ,
           YARG_CP="/content/build:/content/lib/onnxruntime_gpu.jar:/content/lib/json.jar",
           YARG_STEMS="/content/models/htdemucs_6s.onnx", YARG_WORDS="/content/models/wav2vec2.onnx")
# Work on the Colab disk, then copy to Drive (much faster than writing to Drive song by song)
subprocess.run(["rm", "-rf", "/content/out"])
cmd = ["python", "/content/YARG_Resurrected/converter/desktop/batch_convert.py", "/content/songs", "/content/out",
       "--mode", MODE, "--notes", NOTES, "--gpu", "--jobs", JOBS]
with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        print(line, end="")

In [ ]:
#@title 4. Save to Drive (a zip to unzip into Internal storage/YARG/songs on the phone)
import shutil, time, os
stamp = time.strftime("%Y-%m-%d %H%M")
zip_path = shutil.make_archive(os.path.join(OUTPUT, "YARG songs " + stamp), "zip", "/content/out", "Converted")
shutil.copy("/content/out/report.csv", os.path.join(OUTPUT, "report " + stamp + ".csv"))
print("Saved", zip_path)
import pandas as pd
pd.read_csv("/content/out/report.csv")[["file", "folder", "bpm", "tempo", "guitarFrom", "lyrics", "vocalLines", "guitarExpert",
                                       "bassExpert", "deepGuitarExpert", "deepBassExpert", "seconds", "error"]]

**On the phone:** unzip into `Internal storage/YARG/songs/`, then Settings > Songs > Scan Songs.
For the Deep Music Mode parts, turn on the **Deep Music Mode** modifier before playing a song
(guitar and bass then play the fuller parts; a song without them plays as usual).

**report.csv** lists every song: its tempo (and whether it was counted at half or twice the
tempo first read, by the backbeat), what the guitar chart follows, the note counts of the regular
and Deep Music Mode parts, where its lyrics came from (timed online, words online timed from
the voice, written by speech-to-text, or none), how many vocal lines, and anything that failed.
Lyrics written by speech-to-text are worth a look: they are in each song's `lyrics.lrc`.

**5. Optional: check the converter against the test recordings** (about 20 s a song once each has been split; the first run of a song splits it, about half its length on a GPU-less runtime)

In [ ]:
#@title 5. Check the converter (needs your copies of the test recordings)
TEST_SONGS = "/content/drive/MyDrive/YARG/test-songs" #@param {type:"string"}
import os, subprocess
if not os.path.isdir(TEST_SONGS) or not os.listdir(TEST_SONGS):
    print("No test recordings in", TEST_SONGS, "- skipped. The list of recordings is in converter/real-songs/expected.tsv.")
else:
    # Separated instruments are kept in TEST_SONGS/.cache on Drive, so later checks are quick
    env = dict(os.environ, YARG_JAVA_OPTS="-Dyarg.cuda=1", LC_ALL="C.UTF-8")
    cmd = ["bash", "/content/YARG_Resurrected/converter/real-songs/check.sh", TEST_SONGS,
           "/content/models/htdemucs_6s.onnx", "/content/lib/onnxruntime_gpu.jar", "/content/test-out"]
    with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            if line.startswith(("==", "  ok", "  FAIL", "  row", "  tempo", "All ", "FAILED")) or "not in expected" in line:
                print(line, end="")
    print("\nA FAIL means a chart lost something it used to get; 'row:' lines are what was measured (see converter/real-songs/README.md).")
